# Pie chart generator

In [2]:
import json
import shutil
from datetime import datetime
from pathlib import Path
from uuid import uuid4

import altair as alt
import kaleido
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import numpy as np
import pandas as pd

try:
    import seaborn as sns
except Exception:
    sns = None
import plotly.express as px
import vl_convert

# Groq
import os
os.environ["GROQ_API_KEY"] = "YOUR_GROQ_API_KEY_HERE"

from groq import Groq

In [3]:
from pathlib import Path

# Project paths
PROJECT = Path.cwd().resolve()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent

DATA_TRAIN = Path(r"C:\Users\Michelle\I2R\data\train")

# Normal generated outputs
OUT_ROOT = PROJECT / "outputs"  /"generated" / "pieplots"

# Testing outputs
PIE_TESTING_ROOT = PROJECT / "testing" / "pie_testing"

OUT_ROOT.mkdir(parents=True, exist_ok=True)
PIE_TESTING_ROOT.mkdir(parents=True, exist_ok=True)

CLEAR_OUTPUT = True
LIBRARIES = ["altair", "matplotlib", "seaborn", "plotly"]
SUBDIRS = ["images", "tables", "metadata"]

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

# Data

In [4]:
# ── Dataset registry ──────────────────────────────────────────────────────────
# Add new datasets here. Each entry needs:
#   "path"        : Path to the CSV file
#   "numeric_cols": List of numeric columns to use as pie values
#   "group_cols"  : List of categorical columns to use as pie categories
#   "date_col"    : Optional date column name (None if not applicable)
#   "agg_cols"    : Optional list of columns to aggregate by for time-based sampling
#   "loader"      : Optional function name to pre-process the dataframe (or None)
DATASET_REGISTRY = {
    "warehouse_retail": {
        "path": DATA_TRAIN / "Warehouse_and_Retail_Sales.csv",
        "numeric_cols": ["RETAIL SALES", "WAREHOUSE SALES", "RETAIL TRANSFERS"],
        "group_cols":   ["ITEM TYPE", "SUPPLIER"],
        "date_col":     "date",
        "agg_cols":     ["YEAR", "MONTH"],
        "loader":       "load_warehouse_retail",
    },
    # ── Add new datasets below ────────────────────────────────────────────────
    # "my_dataset": {
    #     "path":         DATA_TRAIN / "my_file.csv",
    #     "numeric_cols": ["col_a"],
    #     "group_cols":   ["category"],
    #     "date_col":     None,
    #     "agg_cols":     None,
    #     "loader":       None,
    # },
}

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)

In [5]:
# ── Dataset loaders ───────────────────────────────────────────────────────────

def load_warehouse_retail(df: pd.DataFrame) -> pd.DataFrame:
    df["date"] = pd.to_datetime(
        dict(year=df["YEAR"].astype("Int64"), month=df["MONTH"].astype("Int64"), day=1),
        errors="coerce",
    )
    return df


def get_loader(name: str):
    loaders = {
        "load_warehouse_retail": load_warehouse_retail,
        # Register new loaders here
    }
    return loaders.get(name)


def load_dataset(dataset_name: str) -> pd.DataFrame | None:
    """Load and pre-process a registered dataset. Returns None if file not found."""
    spec = DATASET_REGISTRY.get(dataset_name)
    if spec is None:
        raise KeyError(f"Unknown dataset: {dataset_name}")
    path = spec["path"]
    if not path.exists():
        print(f"Dataset not found: {path}")
        return None
    df = pd.read_csv(path)
    if spec["loader"]:
        loader_fn = get_loader(spec["loader"])
        if loader_fn:
            df = loader_fn(df)
    return df


# Load all available datasets
DATASETS = {}
for ds_name, ds_spec in DATASET_REGISTRY.items():
    ds = load_dataset(ds_name)
    if ds is not None:
        DATASETS[ds_name] = ds
        print(f"Loaded '{ds_name}': {len(ds)} rows, cols: {list(ds.columns)}")

DEFAULT_DATASET = next(iter(DATASETS)) if DATASETS else None
print(f"Default dataset: {DEFAULT_DATASET}")

Loaded 'warehouse_retail': 307645 rows, cols: ['YEAR', 'MONTH', 'SUPPLIER', 'ITEM CODE', 'ITEM DESCRIPTION', 'ITEM TYPE', 'RETAIL SALES', 'RETAIL TRANSFERS', 'WAREHOUSE SALES', 'date']
Default dataset: warehouse_retail


## 1. Sampling weights

Weights are derived directly from observed frequencies in the reference dataset.
Each parameter maps `value → probability`.

In [6]:
SAMPLING_WEIGHTS = {
    "title_present":          {True: 0.71, False: 0.29},
    "title_location":         {"center": 0.56, "right": 0.01, "none": 0.35, "left": 0.08},
    "title_color":            {"black": 0.6582, "gray": 0.2278, "orange": 0.0633, "red": 0.0253, "green": 0.0253},
    "title_size":             {"medium": 0.5316, "small": 0.2785, "large": 0.1899},
    "subtitle_present":       {False: 0.93, True: 0.07},
    "legend_present":         {True: 0.63, False: 0.37},
    "legend_title_size":      {"none": 0.91, "small": 0.09},
    "legend_title_color":     {"black": 1.0},
    "legend_text_color":      {"black": 0.7313, "darkgray": 0.1642, "same_as_title": 0.1045},
    "legend_outline":         {False: 0.87, True: 0.13},
    "legend_fill":            {"none": 0.9636, "gray_block": 0.0364},
    "legend_orientation":     {"none": 0.41, "right": 0.29, "bottom": 0.12, "top_right": 0.10, "top_left": 0.03, "left": 0.03, "top": 0.02},
    "slice_text_present":     {True: 0.8283, False: 0.1717},
    "slice_text_content":     {"percent": 0.32, "none": 0.14, "label_percent_stacked": 0.18, "value": 0.08, "percent_inside_label_outside": 0.06, "label_value_stacked": 0.05, "label_percent_inline": 0.05, "label": 0.04, "label_value_percent_inline": 0.03, "percent_above_label": 0.03, "label_value_inline": 0.01, "percent_box": 0.01},
    "slice_text_orientation": {"horizontal": 0.8586, "none": 0.1111, "radial": 0.0202, "mixed": 0.0101},
    "label_placement":        {"inside": 0.44, "outside_without_leader_lines": 0.18, "outside_with_leader_lines": 0.11, "none": 0.11, "split_label_outside_percent_inside": 0.07, "mixed": 0.04, "fit_inside_else_outside": 0.04, "split_label_outside_with_leader_lines_value_inside": 0.01},
    "label_text_color":       {"black": 0.52, "white": 0.15, "none": 0.11, "adaptive": 0.08, "adaptive_category_black": 0.06, "same_as_slice": 0.03, "same_as_title": 0.02, "white_on_black_box": 0.01, "same_as_legend_text": 0.01, "outside_black_inside_white": 0.01},
    "percentage_format":      {"integer": 0.4646, "none": 0.2626, "one_decimal": 0.1919, "two_decimals": 0.0808},
    "slice_border_width":     {0: 0.43, 1: 0.40, 2: 0.17},
    "slice_border_color":     {"none": 0.43, "white": 0.35, "black": 0.22},
    "outline_chart":          {False: 0.67, True: 0.33},
    "image_outline":          {False: 1.0, True:0.0}, #{False: 0.84, True: 0.16},
    "background":             {"white": 0.88, "transparent": 0.05, "light_color": 0.04, "light_gray": 0.02, "dark_gray": 0.01},
    "palette_type":           {"muted_multicolor": 0.33, "bright_multicolor": 0.23, "random": 0.13, "light_multicolor": 0.08, "rainbow_ordered": 0.06, "dark_monochrome": 0.06, "bright_monochrome": 0.05, "dark_palette": 0.02, "ordered": 0.02, "categorical": 0.01, "one_color": 0.01},
    "sort_order":             {"unsorted": 0.46, "descending": 0.44, "ascending": 0.10},
    "slice_count":            {5: 0.32, 4: 0.17, 6: 0.12, 8: 0.07, 3: 0.06, 10: 0.06, 15: 0.04, 7: 0.04, 12: 0.03, 2: 0.02, 16: 0.02, 17: 0.02, 9: 0.01, 13: 0.01, 14: 0.01},
}

TITLE_COLOR_PALETTES = {
    "black":  ["#000000", "#1a1a1a", "#212121"],
    "gray":   ["#555555", "#6b6b6b", "#888888", "#9e9e9e"],
    "orange": ["#e65100", "#ef6c00", "#f57c00", "#fb8c00", "#ff9800"],
    "red":    ["#b71c1c", "#c62828", "#d32f2f", "#e53935"],
    "green":  ["#1b5e20", "#2e7d32", "#388e3c", "#43a047"],
}

# For each palette type, defines slight variations in the base colors used.
# Each entry is a list of alternative base color sets to randomly pick from.
PALETTE_VARIATION = {
    "muted_multicolor": [
        ["#7b9acc", "#d4a373", "#84a98c", "#b5838d", "#9f86c0", "#8d99ae", "#6c9a8b"],
        ["#6b8cba", "#c8956b", "#74997c", "#a5737d", "#8f76b0", "#7d899e", "#5c8a7b"],
        ["#8baad4", "#e4b383", "#94b99c", "#c5939d", "#af96d0", "#9da9be", "#7caa9b"],
        ["#7090b8", "#cc9060", "#70957a", "#b07880", "#9070b0", "#808898", "#609080"],
    ],
    "bright_multicolor": [
        ["#ff595e", "#ffca3a", "#8ac926", "#1982c4", "#6a4c93", "#ff924c", "#00b4d8"],
        ["#e84855", "#f7b731", "#78b522", "#1572b0", "#5a3c83", "#e8823c", "#00a4c8"],
        ["#ff6b6b", "#ffd93d", "#9de24f", "#2196f3", "#7c57a3", "#ffa45b", "#00c8e8"],
        ["#ff4040", "#ffbb00", "#7ab820", "#1060a0", "#603090", "#ff8030", "#00a0c0"],
    ],
    "light_multicolor": [
        ["#ffd6e0", "#d6f5e3", "#dbeafe", "#fff3bf", "#f3d9fa", "#ffe8cc", "#d0ebff"],
        ["#ffc6d0", "#c6e8d3", "#cbdefe", "#ffebb0", "#e8c9ea", "#ffd8bc", "#c0dbff"],
        ["#ffe6f0", "#e6fff3", "#ebf4ff", "#fffbcf", "#f9e9ff", "#fff8dc", "#e0f5ff"],
        ["#ffb0c8", "#b0ddc0", "#b8d4f8", "#ffe098", "#dbb8e0", "#ffc8a8", "#b8d0f8"],
    ],
    "bright_monochrome": [
        {"cmap": "YlOrBr", "low": 0.35, "high": 0.90},
        {"cmap": "Oranges", "low": 0.40, "high": 0.90},
        {"cmap": "YlOrRd", "low": 0.30, "high": 0.85},
        {"cmap": "OrRd",   "low": 0.35, "high": 0.85},
    ],
    "dark_monochrome": [
        {"cmap": "Blues",   "low": 0.45, "high": 0.90},
        {"cmap": "Purples", "low": 0.40, "high": 0.90},
        {"cmap": "Greens",  "low": 0.40, "high": 0.85},
        {"cmap": "Greys",   "low": 0.35, "high": 0.80},
    ],
    "rainbow_ordered": [
        {"cmap": "rainbow",  "low": 0.0, "high": 1.0},
        {"cmap": "hsv",      "low": 0.0, "high": 0.85},
        {"cmap": "gist_rainbow", "low": 0.0, "high": 1.0},
        {"cmap": "turbo",    "low": 0.05, "high": 0.95},
    ],
    "ordered": [
        {"cmap": "viridis", "low": 0.10, "high": 0.90},
        {"cmap": "plasma",  "low": 0.10, "high": 0.90},
        {"cmap": "cividis", "low": 0.10, "high": 0.90},
        {"cmap": "magma",   "low": 0.15, "high": 0.90},
    ],
    "dark_palette": [
        ["#1b263b", "#415a77", "#5c677d", "#7d8597", "#9aa6b2", "#3a0ca3", "#560bad"],
        ["#0d1b2a", "#1b3a5c", "#2e5f8a", "#4a7fa5", "#6a9fbf", "#2a0a8a", "#440993"],
        ["#2b2d42", "#5c4a72", "#7b6fa0", "#8d99ae", "#adb5bd", "#4a0ca3", "#6a0dad"],
        ["#1a1a2e", "#16213e", "#0f3460", "#533483", "#e94560", "#2a0a7a", "#3a0a93"],
    ],
    "one_color": [
        "#1f77b4", "#2ca02c", "#d62728", "#9467bd",
        "#8c564b", "#e377c2", "#17becf", "#ff7f0e",
    ],
    "categorical": [
        "tab10", "tab20", "Set1", "Set2", "Set3", "Paired",
    ],
    "random": [
        ["#1f77b4","#ff7f0e","#2ca02c","#d62728","#9467bd","#8c564b","#e377c2","#7f7f7f","#bcbd22","#17becf"],
        ["#4e79a7","#f28e2b","#e15759","#76b7b2","#59a14f","#edc948","#b07aa1","#ff9da7","#9c755f","#bab0ac"],
        ["#1abc9c","#2ecc71","#3498db","#9b59b6","#e74c3c","#f39c12","#16a085","#27ae60","#2980b9","#8e44ad"],
        ["#264653","#2a9d8f","#e9c46a","#f4a261","#e76f51","#457b9d","#1d3557","#a8dadc","#f1faee","#e63946"],
    ],
}

TITLE_SIZE_PALETTES = {
    "small":  [9, 10, 11, 12],
    "medium": [13, 14, 15, 16],
    "large":  [18, 20, 22, 24],
}

LEGEND_FILL_PALETTES = {
    "gray_block": ["#e9ecef60", "#dee2e694", "#ced4daae", "#d8d8d8d8", "#e0e0e0da", "#f0f0f0"],
}

# For multi-token slice text contents, defines which separator variants are possible.
# "\n" = stacked (one per line), " " = side by side inline
SLICE_TEXT_LAYOUT_VARIANTS = {
    "label_value_inline":              ["\n", " "],
    "label_percent_inline":            ["\n", " "],
    "label_value_percent_inline":      ["\n", " "],
    "label_value_stacked":             ["\n", " "],
    "label_percent_stacked":           ["\n", " "],
    "percent_above_label":             ["\n", " "],
}

LABEL_TEXT_COLOR_PALETTES = {
    "black":   ["#000000", "#1a1a1a", "#212121", "#2d2d2d"],
    "white":   ["#ffffff", "#f5f5f5", "#fafafa"],
    "adaptive": ["adaptive"],          # resolved at render time, no subset needed
    "adaptive_category_black": ["adaptive_category_black"],
    "same_as_slice":      ["same_as_slice"],
    "same_as_title":      ["same_as_title"],
    "same_as_legend_text":["same_as_legend_text"],
    "outside_black_inside_white": ["outside_black_inside_white"],
    "white_on_black_box": ["white_on_black_box"],
    "none":    ["none"],
}

SLICE_BORDER_WIDTH_PALETTES = {
    0: [0],
    1: [0.5, 0.75, 1.0, 1.25, 1.5],
    2: [1.75, 2.0, 2.25, 2.5, 3.0],
}

OUTLINE_CHART_WIDTH_PALETTES = {
    True:  [0.8, 1.0, 1.2, 1.5, 2.0, 2.5],
    False: [0],
}

BACKGROUND_PALETTES = {
    "white":       ["#ffffff", "#fefefe", "#fdfdfd", "#f9f9f9"],
    "transparent": ["transparent"],
    "light_gray":  ["#f1f3f5", "#e9ecef", "#dee2e6", "#ced4da"],
    "dark_gray":   ["#343a40", "#495057", "#3d3d3d", "#2b2b2b"],
    "light_color": ["#f3f0ff", "#fff0f6", "#e8f4fd", "#f0fff4", "#fffbe6", "#fff4e6"],
}





In [7]:
def normalize_weights(weights: dict) -> dict:
    """Ensure each parameter's weights sum to 1.0."""
    out = {}
    for param, codes in weights.items():
        total = sum(codes.values())
        out[param] = {k: v / total for k, v in codes.items()}
    return out


def sample_style(rng: np.random.Generator, weights: dict) -> dict:
    style = {}
    for param, val_probs in weights.items():
        vals = list(val_probs.keys())
        probs = np.array(list(val_probs.values()), dtype=float)
        probs /= probs.sum()
        style[param] = vals[int(rng.choice(len(vals), p=probs))]

    # Expand title_color group name → specific hex
    color_group = style.get("title_color")
    if color_group in TITLE_COLOR_PALETTES:
        palette = TITLE_COLOR_PALETTES[color_group]
        style["title_color"] = palette[int(rng.integers(0, len(palette)))]

    # Expand title_size group name → specific int (pt)
    size_group = style.get("title_size")
    if size_group in TITLE_SIZE_PALETTES:
        palette = TITLE_SIZE_PALETTES[size_group]
        style["title_size"] = palette[int(rng.integers(0, len(palette)))]

    # Expand legend_fill group name → specific hex
    fill_group = style.get("legend_fill")
    if fill_group in LEGEND_FILL_PALETTES:
        palette = LEGEND_FILL_PALETTES[fill_group]
        style["legend_fill"] = palette[int(rng.integers(0, len(palette)))]

    # Expand label_text_color group name → specific value
    label_color_group = style.get("label_text_color")
    if label_color_group in LABEL_TEXT_COLOR_PALETTES:
        palette = LABEL_TEXT_COLOR_PALETTES[label_color_group]
        style["label_text_color"] = palette[int(rng.integers(0, len(palette)))]

    # Expand slice_border_width group → specific float
    border_group = style.get("slice_border_width")
    if border_group in SLICE_BORDER_WIDTH_PALETTES:
        palette = SLICE_BORDER_WIDTH_PALETTES[border_group]
        style["slice_border_width"] = palette[int(rng.integers(0, len(palette)))]

    # Expand outline_chart → specific float linewidth
    outline_group = style.get("outline_chart")
    if outline_group in OUTLINE_CHART_WIDTH_PALETTES:
        palette = OUTLINE_CHART_WIDTH_PALETTES[outline_group]
        style["outline_chart"] = palette[int(rng.integers(0, len(palette)))]

    # Expand background group → specific color
    bg_group = style.get("background")
    if bg_group in BACKGROUND_PALETTES:
        palette = BACKGROUND_PALETTES[bg_group]
        style["background"] = palette[int(rng.integers(0, len(palette)))]

    # Store palette variation seed for use in build_palette
    palette_type = style.get("palette_type")
    if palette_type in PALETTE_VARIATION:
        variants = PALETTE_VARIATION[palette_type]
        style["palette_variation_idx"] = int(rng.integers(0, len(variants)))
    else:
        style["palette_variation_idx"] = 0

    # Randomly pick separator variant for multi-token slice text layouts
    content = style.get("slice_text_content")
    if content in SLICE_TEXT_LAYOUT_VARIANTS:
        variants = SLICE_TEXT_LAYOUT_VARIANTS[content]
        style["slice_text_sep"] = variants[int(rng.integers(0, len(variants)))]
    else:
        style["slice_text_sep"] = None

    return style

## 2. Data sampling

In [8]:
def cap_visible_slices(series: pd.Series, visible_slice_count: int) -> pd.Series | None:
    s = series[series > 0].sort_values(ascending=False)
    if len(s) < 2:
        return None
    visible_slice_count = int(max(2, min(visible_slice_count, len(s))))
    if len(s) <= visible_slice_count:
        return s.copy()
    keep = max(1, visible_slice_count - 1)
    top = s.head(keep).copy()
    other_sum = float(s.iloc[keep:].sum())
    if other_sum > 0:
        top.loc["Other"] = other_sum
    return top


def sample_pie_data_from_df(
    df: pd.DataFrame,
    spec: dict,
    rng: np.random.Generator,
    style: dict,
    min_total: float = 1.0,
) -> tuple[pd.DataFrame, dict] | None:
    numeric_cols = spec["numeric_cols"]
    group_cols   = spec["group_cols"]
    date_col     = spec.get("date_col")
    slice_count  = int(style.get("slice_count", 5))

    value_col    = str(rng.choice(numeric_cols))
    category_col = str(rng.choice(group_cols))

    strategy = str(rng.choice(["group", "time_window", "filtered_group"],
                               p=[0.40, 0.35, 0.25]))
    context = {"value_col": value_col, "strategy": strategy}

    if strategy == "time_window" and date_col and date_col in df.columns:
        window_months = int(rng.choice([1, 3, 6, 12], p=[0.30, 0.35, 0.20, 0.15]))
        monthly_totals = df.groupby(date_col)[value_col].sum()
        valid_months = monthly_totals[monthly_totals > min_total].index.to_numpy()
        if len(valid_months) == 0:
            return None
        start = pd.to_datetime(rng.choice(valid_months))
        end   = start + pd.DateOffset(months=window_months)
        dfw   = df[(df[date_col] >= start) & (df[date_col] < end)].copy()
        if len(dfw) == 0:
            return None
        series = dfw.groupby(category_col)[value_col].sum()
        context.update({"category_col": category_col, "start": start,
                        "end": end, "window_months": window_months})

    elif strategy == "filtered_group" and len(group_cols) >= 2:
        other_cols = [c for c in group_cols if c != category_col]
        filter_col = str(rng.choice(other_cols))
        # Only pick filter values that produce enough rows and non-zero totals
        candidates = np.array(df[filter_col].dropna().unique(), dtype=str)
        rng.shuffle(candidates)
        dfw = None
        for val in candidates[:10]:
            tmp = df[df[filter_col] == val].copy()
            tmp_series = tmp.groupby(category_col)[value_col].sum()
            tmp_series = tmp_series[tmp_series > 0]
            if len(tmp_series) >= 2 and tmp_series.sum() > min_total:
                dfw = tmp
                filter_val = str(val)
                break
        if dfw is None:
            return None
        series = dfw.groupby(category_col)[value_col].sum()
        context.update({"category_col": category_col, f"filter_{filter_col}": filter_val})

    else:  # "group"
        series = df.groupby(category_col)[value_col].sum()
        context["category_col"] = category_col

    # Keep only positive values — no percentage pre-filter
    series = series[series > 0]
    if len(series) < 2:
        return None

    # cap_visible_slices handles the tail via "Other"
    visible_series = cap_visible_slices(series, slice_count)
    if visible_series is None or len(visible_series) < 2:
        return None

    total = float(visible_series.sum())
    if total <= min_total:
        return None

    # No dominance check — "Other" being large is valid for real data
    plot_df = visible_series.reset_index()
    plot_df.columns = [category_col, value_col]

    context.update({
        "category_col":          category_col,
        "n_slices":              int(len(plot_df)),
        "total":                 total,
        "has_other_category":    bool("Other" in plot_df[category_col].values),
        "requested_slice_count": slice_count,
    })

    return plot_df, context

## 3. Shared styling helpers

In [9]:
CONTENT_LAYOUTS = {
    "none":                       {"tokens": [], "sep": ""},
    "percent":                    {"tokens": ["percent"], "sep": ""},
    "percent_box":                {"tokens": ["percent"], "sep": "", "boxed": True},
    "label":                      {"tokens": ["label"], "sep": ""},
    "value":                      {"tokens": ["value"], "sep": ""},
    "label_value_inline":         {"tokens": ["label", "value"],   "template": "{label} ({value})"},
    "label_percent_inline":       {"tokens": ["label", "percent"], "template": "{label} ({percent})"},
    "label_value_percent_inline": {"tokens": ["label", "value", "percent"], "template": "{label}: {value} ({percent})"},
    "label_value_stacked":        {"tokens": ["label", "value"],   "sep": "\n"},
    "label_percent_stacked":      {"tokens": ["label", "percent"], "sep": "\n"},
    "percent_above_label":        {"tokens": ["percent", "label"], "sep": "\n"},
    "percent_inside_label_outside": {
        "inside_tokens": ["percent"], "outside_tokens": ["label"],
        "inside_sep": "", "outside_sep": "",
    },
}


def rgba_to_hex(color) -> str:
    return mcolors.to_hex(color, keep_alpha=False)


def relative_luminance(color) -> float:
    rgb = mcolors.to_rgb(color)
    return 0.2126 * rgb[0] + 0.7152 * rgb[1] + 0.0722 * rgb[2]


def choose_contrast_text_color(fill_color: str) -> str:
    return "black" if relative_luminance(fill_color) >= 0.55 else "white"


def value_to_display_text(value: float) -> str:
    if abs(value - round(value)) < 1e-9:
        return f"{int(round(value))}"
    if abs(value) >= 100:
        return f"{value:,.0f}"
    return f"{value:,.1f}"


def get_background_color(style: dict) -> str:
    bg = style.get("background", "#ffffff")
    if bg in {"transparent", "none"}:
        return "none"
    # If it's already a hex or named color, pass through directly
    try:
        mcolors.to_rgb(bg)
        return bg
    except Exception:
        return "#ffffff"

def get_title_fontsize(style: dict) -> int:
    size = style.get("title_size", 16)
    if isinstance(size, int):
        return size
    return {"small": 12, "medium": 16, "large": 20}.get(size, 16)


def get_title_alignment(style: dict) -> tuple[float, str]:
    location = style.get("title_location", "center")
    if location == "left":  return 0.01, "left"
    if location == "right": return 0.99, "right"
    return 0.50, "center"


def get_altair_title_anchor(style: dict) -> str:
    location = style.get("title_location", "center")
    if location == "left":  return "start"
    if location == "right": return "end"
    return "middle"


def get_plotly_title_anchor(style: dict) -> tuple[float, str]:
    location = style.get("title_location", "center")
    if location == "left":  return 0.01, "left"
    if location == "right": return 0.99, "right"
    return 0.50, "center"


def get_legend_title_text(category_col: str, style: dict) -> str | None:
    return category_col if style.get("legend_title_size") != "none" else None


def get_legend_text_color(style: dict) -> str | None:
    # If legend has a filled background, always use black for readability
    fill = style.get("legend_fill", "none")
    if fill not in {"none", None}:
        return "black"
    mode = style.get("legend_text_color")
    if mode is None: return None
    if mode == "same_as_title": return style.get("title_color", "black")
    return mode


def build_palette(n: int, style: dict, rng: np.random.Generator | None = None) -> list[str]:
    palette_type = style.get("palette_type", "random")
    var_idx      = style.get("palette_variation_idx", 0)
    rng          = rng or np.random.default_rng()

    def from_cmap(spec):
        cmap = plt.get_cmap(spec["cmap"])
        return [rgba_to_hex(cmap(spec["low"] + (spec["high"] - spec["low"]) * i / max(1, n - 1))) for i in range(n)]

    def from_list(base):
        return [base[i % len(base)] for i in range(n)]

    variants = PALETTE_VARIATION.get(palette_type)

    if palette_type == "categorical":
        cmap_name = variants[var_idx % len(variants)] if variants else "tab10"
        cmap = plt.get_cmap(cmap_name)
        return [rgba_to_hex(cmap(i % cmap.N)) for i in range(n)]

    if palette_type == "one_color":
        base = variants[var_idx % len(variants)] if variants else "#1f77b4"
        rgb  = np.array(mcolors.to_rgb(base))
        out  = []
        for i in range(n):
            mix     = 0.20 + 0.60 * i / max(1, n - 1)
            blended = rgb * (0.60 + 0.30 * mix) + (1 - rgb) * (0.15 + 0.15 * i / max(1, n - 1))
            out.append(rgba_to_hex(np.clip(blended, 0, 1)))
        return out

    if palette_type == "random":
        base = variants[var_idx % len(variants)] if variants else []
        idx  = rng.choice(len(base), size=n, replace=n > len(base))
        return [base[i] for i in idx]

    if palette_type in {"bright_monochrome", "dark_monochrome", "rainbow_ordered", "ordered"}:
        spec = variants[var_idx % len(variants)] if variants else {"cmap": "tab10", "low": 0, "high": 1}
        return from_cmap(spec)

    if palette_type in {"muted_multicolor", "bright_multicolor", "light_multicolor", "dark_palette"}:
        base = variants[var_idx % len(variants)] if variants else []
        return from_list(base)

    if palette_type == "grayscale":
        cmap = plt.get_cmap("Greys")
        return [rgba_to_hex(cmap(0.20 + 0.65 * i / max(1, n - 1))) for i in range(n)]

    # fallback
    return [rgba_to_hex(plt.get_cmap("tab10")(i % 10)) for i in range(n)]

def resolve_border_settings(style: dict) -> tuple[str | None, int]:
    if style.get("slice_border_width", 0) <= 0 or style.get("slice_border_color") == "none":
        return None, 0
    return style.get("slice_border_color", "white"), int(style.get("slice_border_width", 1))


def compute_label_fragments(plot_df: pd.DataFrame, category_col: str, value_col: str, style: dict) -> pd.DataFrame:
    df = plot_df.copy().reset_index(drop=True)
    total = float(df[value_col].sum())
    df["_share"] = df[value_col] / total
    df["_label_text"] = df[category_col].astype(str)
    df["_value_text"] = df[value_col].astype(float).map(value_to_display_text)
    fmt = style.get("percentage_format", "integer")
    if fmt == "integer":
        df["_percent_text"] = (100 * df["_share"]).round(0).astype(int).astype(str) + "%"
        df["_percent_text"] = df["_percent_text"].replace("0%", "")
    elif fmt == "one_decimal":
        df["_percent_text"] = (100 * df["_share"]).round(1).map(lambda x: f"{x:.1f}%" if x >= 0.05 else "")
    elif fmt == "two_decimals":
        df["_percent_text"] = (100 * df["_share"]).round(2).map(lambda x: f"{x:.2f}%" if x >= 0.005 else "")
    else:
        df["_percent_text"] = ""
    return df


def compose_text(df: pd.DataFrame, tokens: list[str], sep: str = "", template: str | None = None, style: dict | None = None) -> pd.Series:
    if not tokens:
        return pd.Series([""] * len(df), index=df.index)
    if template:
        rows = []
        for _, row in df.iterrows():
            values = {token: str(row.get(f"_{token}_text", "")) for token in tokens}
            rows.append(template.format(**values))
        return pd.Series(rows, index=df.index)
    # Use sampled separator if available, otherwise fall back to layout default
    if style is not None and style.get("slice_text_sep") is not None:
        sep = style["slice_text_sep"]
    cols = [f"_{token}_text" for token in tokens]
    result = df[cols[0]].astype(str).copy()
    for col in cols[1:]:
        result = result + sep + df[col].astype(str)
    return result

def build_label_plan(plot_df: pd.DataFrame, category_col: str, value_col: str, style: dict) -> pd.DataFrame:
    df = compute_label_fragments(plot_df, category_col, value_col, style)
    placement = style.get("label_placement", "none")
    content   = style.get("slice_text_content", "none")

    df["_inside_text"]         = ""
    df["_outside_text"]        = ""
    df["_outside_with_leader"] = False
    df["_boxed_text"]          = False

    layout = CONTENT_LAYOUTS.get(content, CONTENT_LAYOUTS["none"])

    if content == "none" or placement == "none":
        return df

    if "inside_tokens" in layout or "outside_tokens" in layout:
        df["_inside_text"]  = compose_text(df, layout.get("inside_tokens", []),  layout.get("inside_sep", ""),  style=style)
        df["_outside_text"] = compose_text(df, layout.get("outside_tokens", []), layout.get("outside_sep", ""), style=style)
        df["_outside_with_leader"] = placement in {"outside_with_leader_lines", "split_label_outside_with_leader_lines_value_inside"}
        return df

    full_text = compose_text(df, layout.get("tokens", []), layout.get("sep", ""), layout.get("template"), style=style)
    boxed = bool(layout.get("boxed", False))

    if placement == "inside":
        df["_inside_text"] = full_text; df["_boxed_text"] = boxed
    elif placement == "outside_without_leader_lines":
        df["_outside_text"] = full_text
    elif placement == "outside_with_leader_lines":
        df["_outside_text"] = full_text; df["_outside_with_leader"] = True
    elif placement == "mixed":
        m = df["_share"] >= 0.12
        df.loc[m,  "_inside_text"]  = full_text[m];  df.loc[m,  "_boxed_text"] = boxed
        df.loc[~m, "_outside_text"] = full_text[~m]
    elif placement == "fit_inside_else_outside":
        m = (df["_share"] >= 0.10) & (full_text.str.len() <= 14)
        df.loc[m,  "_inside_text"]  = full_text[m];  df.loc[m,  "_boxed_text"] = boxed
        df.loc[~m, "_outside_text"] = full_text[~m]
    elif placement == "split_label_outside_percent_inside":
        df["_inside_text"]  = df["_percent_text"]
        df["_outside_text"] = df["_label_text"]
    elif placement == "split_label_outside_with_leader_lines_value_inside":
        df["_inside_text"]  = df["_value_text"]
        df["_outside_text"] = df["_label_text"]
        df["_outside_with_leader"] = True
    else:
        df["_inside_text"] = full_text; df["_boxed_text"] = boxed

    return df

def resolve_text_color(mode: str, zone: str, fill_color: str, style: dict) -> str:
    legend_text_color = get_legend_text_color(style)
    title_color       = style.get("title_color", "black")

    # If text is in a box, check the box background color
    # For white_on_black_box the box is always black → always white text
    if mode == "white_on_black_box":
        return "white"

    # If the label has a box (boxed=True from build_label_plan), the box color
    # is the slice fill. If that fill is dark, force white text regardless of mode.
    # This is signalled by zone == "inside_boxed"
    if zone == "inside_boxed":
        return "white" if relative_luminance(fill_color) < 0.45 else "black"

    # Outside labels always on background — never white or near-white
    if zone == "outside":
        if mode in {"white", "same_as_slice", "adaptive", "adaptive_category_black"}:
            return "black"
        if mode == "same_as_title":      return title_color if relative_luminance(title_color) <= 0.85 else "black"
        if mode == "same_as_legend_text": return legend_text_color or "black"
        if mode == "outside_black_inside_white": return "black"
        # If mode is a hex color, check it's visible on white background
        try:
            if relative_luminance(mode) > 0.85:
                return "black"
            return mode
        except Exception:
            return "black"

    # Inside labels
    if mode == "black":               return "black"
    if mode == "white":               return "white"
    if mode == "same_as_slice":       return fill_color
    if mode == "same_as_title":       return title_color
    if mode == "same_as_legend_text": return legend_text_color or "black"
    if mode == "adaptive":            return choose_contrast_text_color(fill_color)
    if mode == "adaptive_category_black": return choose_contrast_text_color(fill_color)
    if mode == "outside_black_inside_white": return "white"
    # Hex color passed directly
    try:
        mcolors.to_rgb(mode)
        return mode
    except Exception:
        return "black"

def apply_sort_order(plot_df: pd.DataFrame, value_col: str, style: dict, rng: np.random.Generator | None = None) -> pd.DataFrame:
    sort_order = style.get("sort_order", "descending")
    if sort_order == "descending": return plot_df.sort_values(value_col, ascending=False).reset_index(drop=True)
    if sort_order == "ascending":  return plot_df.sort_values(value_col, ascending=True).reset_index(drop=True)
    rng = rng or np.random.default_rng()
    seed = int(rng.integers(0, 1_000_000))
    return plot_df.sample(frac=1.0, random_state=seed).reset_index(drop=True)


def harmonize_style(style: dict) -> dict:
    style = dict(style)

    if (not style.get("title_present", True)) or style.get("title_location") == "none":
        style.update({"title_present": False, "title_location": "none", "subtitle_present": False})

    if not style.get("legend_present", False) or style.get("legend_orientation") == "none":
        style.update({"legend_present": False, "legend_orientation": "none",
                      "legend_title_size": "none", "legend_title_color": None,
                      "legend_text_color": None, "legend_outline": False, "legend_fill": "none"})

    if style.get("legend_title_size") == "none":
        style["legend_title_color"] = None

    no_text = (
        style.get("slice_text_orientation") == "none" or
        style.get("label_text_color") == "none" or
        style.get("label_placement") == "none" or
        style.get("slice_text_content") == "none"
    )
    if no_text:
        style.update({"slice_text_present": False, "slice_text_content": "none",
                      "label_placement": "none", "slice_text_orientation": "none"})

    if style.get("percentage_format") == "none" and style.get("slice_text_content") in {
        "percent", "percent_box", "label_percent_inline", "label_percent_stacked",
        "percent_above_label", "percent_inside_label_outside", "label_value_percent_inline",
    }:
        style.update({"slice_text_content": "none", "slice_text_present": False,
                      "label_placement": "none", "slice_text_orientation": "none"})

    if style.get("slice_border_width", 0) == 0:
        style["slice_border_color"] = "none"

    slice_count = style.get("slice_count", 5)
    style["slice_count"] = int(slice_count) if slice_count and int(slice_count) >= 2 else 5

    return style

## 4. Title generation

In [10]:
_groq_client = Groq()  # reads GROQ_API_KEY from env

MAX_TITLE_CHARS = 60

def make_title(context: dict, style: dict | None = None) -> tuple[str, str | None]:
    value_col    = context.get("value_col", "Value")
    category_col = context.get("category_col", "Category")

    filter_parts = []
    for k, v in context.items():
        if k.startswith("filter_"):
            filter_parts.append(f"{k[7:]}: {v}")
    filter_desc = f" (filtered to {', '.join(filter_parts)})" if filter_parts else ""

    time_desc = ""
    if "start" in context:
        start_str = pd.to_datetime(context["start"]).strftime("%B %Y")
        end_str   = pd.to_datetime(context["end"]).strftime("%B %Y")
        time_desc = f", covering {start_str} to {end_str}"

    need_subtitle = style is not None and style.get("subtitle_present", False)

    system_prompt = (
        "You generate short, realistic chart titles for pie charts — the kind you'd see "
        "in a business report or dashboard. Keep titles under 60 characters. "
        "Be concise and natural. No quotes, no markdown."
    )
    user_prompt = (
        f"Generate a chart title for a pie chart showing the breakdown of {value_col} "
        f"by {category_col}{filter_desc}{time_desc}.\n"
    )
    if need_subtitle:
        user_prompt += (
            "Also generate a short subtitle (one line, adds context or time range detail). "
            "Respond in this exact format:\nTITLE: <title here>\nSUBTITLE: <subtitle here>"
        )
    else:
        user_prompt += "Respond with just the title, nothing else."

    try:
        response = _groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_prompt},
            ],
            temperature=0.8,
            max_tokens=80,
        )
        raw = response.choices[0].message.content.strip()

        if need_subtitle and "TITLE:" in raw:
            lines = {k.strip(): v.strip() for k, v in
                     (line.split(":", 1) for line in raw.splitlines() if ":" in line)}
            return lines.get("TITLE", raw)[:MAX_TITLE_CHARS], lines.get("SUBTITLE", "")[:MAX_TITLE_CHARS]

        return raw[:MAX_TITLE_CHARS], None

    except Exception:
        fallback_title = f"{value_col} by {category_col}"[:MAX_TITLE_CHARS]
        fallback_sub = "Business report" if need_subtitle else None
        return fallback_title, fallback_sub

## 5. Pie renderers

In [11]:
def _enforce_fit_inside(plot_df: pd.DataFrame, style: dict) -> dict:
    if style.get("label_placement") != "inside":
        return style
    if style.get("slice_text_content") in {"none", None}:
        return style

    value_col   = plot_df.columns[1]
    label_col   = plot_df.columns[0]
    total       = float(plot_df[value_col].sum())
    shares      = plot_df[value_col] / total
    n           = len(plot_df)
    orientation = style.get("slice_text_orientation", "horizontal")
    content     = style.get("slice_text_content", "none")

    label_tokens = {
        "label", "label_value_inline", "label_percent_inline",
        "label_value_percent_inline", "label_value_stacked",
        "label_percent_stacked", "percent_above_label",
    }
    has_label_token = content in label_tokens
    max_label_len   = int(plot_df[label_col].astype(str).str.len().max())

    # Minimum slice share needed to fit a label inside, by orientation
    # Radial is more forgiving (text runs along the radius) so threshold is lower
    if orientation == "radial":
        share_limit = 0.07 if n <= 5 else 0.10
        char_limit  = 18   if n <= 5 else 14   # radial can handle longer strings
    elif orientation in {"horizontal", "mixed"}:
        share_limit = 0.12 if n <= 5 else 0.15
        char_limit  = 12   if n <= 5 else 9
    else:
        share_limit = 0.10
        char_limit  = 12

    # For horizontal/mixed: also account for label width vs slice angular width.
    # A slice with share s subtends angle 2*pi*s radians. At inner radius ~0.58r,
    # the chord length ≈ 2 * 0.58 * sin(pi*s). We compare this to an estimate of
    # the rendered text width (approx 7px per char at fontsize 10).
    any_overflow = False

    if orientation in {"horizontal", "mixed"} and has_label_token:
        for share in shares:
            chord_px = 2 * 0.58 * 160 * np.sin(np.pi * float(share))  # 160 = outer_radius
            needed_px = max_label_len * 7
            if needed_px > chord_px * 0.80:   # 80% of chord to leave margin
                any_overflow = True
                break

    # Share-based check: any slice too small regardless of orientation
    if (shares < share_limit).any():
        any_overflow = True

    # Character-length check
    if has_label_token and max_label_len > char_limit:
        any_overflow = True

    if any_overflow:
        style = dict(style)
        style["label_placement"] = "outside_with_leader_lines"

    return style


def sanitize_style_for_altair(plot_df: pd.DataFrame, style: dict) -> dict:
    style = dict(style)
    n = len(plot_df)
    if n >= 10:
        style.update({"label_placement": "none", "slice_text_content": "none",
                      "slice_text_present": False, "slice_text_orientation": "none"})
        return harmonize_style(style)
    if style.get("label_placement") == "mixed":
        style["label_placement"] = "fit_inside_else_outside"
    if style.get("slice_text_orientation") == "mixed":
        style["slice_text_orientation"] = "horizontal"
    if style.get("slice_text_orientation") == "radial":
        style["slice_text_orientation"] = "horizontal"
    if style.get("label_placement") == "outside_with_leader_lines" and n >= 7:
        style["label_placement"] = "outside_without_leader_lines"
    style = _enforce_fit_inside(plot_df, style)
    # _enforce_fit_inside may push to outside_with_leader_lines — keep it,
    # but downgrade to without_leader_lines for large n to avoid clutter
    if style.get("label_placement") == "outside_with_leader_lines" and n >= 7:
        style["label_placement"] = "outside_without_leader_lines"
    return harmonize_style(style)

def sanitize_style_for_matplotlib(plot_df: pd.DataFrame, style: dict) -> dict:
    style = dict(style)
    n = len(plot_df)
    total = plot_df.iloc[:, 1].sum()
    shares = plot_df.iloc[:, 1] / total
    if shares.max() > 0.50 and style.get("label_placement") == "inside":
        style["label_placement"] = "outside_with_leader_lines"
    if n >= 12:
        style.update({"label_placement": "none", "slice_text_content": "none",
                      "slice_text_present": False, "slice_text_orientation": "none"})
        return harmonize_style(style)
    if n >= 9 and style.get("label_placement") == "inside":
        style["label_placement"] = "fit_inside_else_outside"
    if n >= 8 and style.get("slice_text_content") in {
        "label_value_percent_inline", "label_value_inline",
        "label_value_stacked", "label_percent_stacked"
    }:
        style["slice_text_content"] = "percent"
    # Always use leader lines for outside placement — prevents overlaps
    if style.get("label_placement") == "outside_without_leader_lines":
        style["label_placement"] = "outside_with_leader_lines"
    style = _enforce_fit_inside(plot_df, style)
    # _enforce_fit_inside may have pushed to outside — ensure leader lines there too
    if style.get("label_placement") == "outside_without_leader_lines":
        style["label_placement"] = "outside_with_leader_lines"
    return harmonize_style(style)


def sanitize_style_for_plotly(plot_df: pd.DataFrame, style: dict) -> dict:
    style = dict(style)
    n = len(plot_df)
    if n >= 12:
        style.update({"label_placement": "none", "slice_text_content": "none",
                      "slice_text_present": False, "slice_text_orientation": "none"})
        return harmonize_style(style)
    if style.get("slice_text_orientation") in {"mixed", "radial"}:
        style["slice_text_orientation"] = "horizontal"
    style = _enforce_fit_inside(plot_df, style)
    return harmonize_style(style)

In [12]:
def _vegalite_angles(value_col_data):
    """
    Compute start/end/mid angles using the same row order that we force
    Vega-Lite to use via the _pie_order channel.

    theta=0 is at 12 o'clock, increasing clockwise.
    """
    vals = np.array(value_col_data, dtype=float)
    total = vals.sum()

    if total <= 0:
        frac = np.ones(len(vals)) / len(vals)
    else:
        frac = vals / total

    start = np.concatenate([[0], np.cumsum(frac[:-1])]) * 2 * np.pi
    end = np.cumsum(frac) * 2 * np.pi
    mid = (start + end) / 2

    return start, end, mid


def compute_altair_text_geometry(
    label_df,
    value_col,
    position,
    outer_radius=160,
    cx=200,
    cy=200,
):
    """
    Compute text positions in the same order as the arcs.

    The key fix is that label_df must already be in _pie_order order,
    and the arc layer must also encode order='_pie_order:Q'.
    """
    df = label_df.copy().sort_values("_pie_order").reset_index(drop=True)

    _, _, mid = _vegalite_angles(df[value_col])

    # Vega-Lite pie coordinates:
    # theta=0 at top, clockwise
    # x = cx + r * sin(theta)
    # y = cy - r * cos(theta)
    sx = np.sin(mid)
    sy = -np.cos(mid)

    if position == "inside":
        # Fixed radius keeps percent labels on the slices instead of
        # clustering near the center for small categories.
        r = outer_radius * 0.67
    else:
        # Outside category labels.
        r = outer_radius * 1.18

    df["_label_x"] = cx + r * sx
    df["_label_y"] = cy + r * sy
    df["_cos"] = sx
    df["_text_angle"] = np.degrees(mid)

    return df


def compute_altair_leaderline_geometry(
    label_df,
    value_col,
    outer_radius=160,
    cx=200,
    cy=200,
    radial_line_len=8,
    horizontal_line_len=14,
    label_pad=5,
):
    """
    Compute outside label and leader-line geometry.

    This also uses _pie_order so the leader lines match the slices.
    """
    df = label_df.copy().sort_values("_pie_order").reset_index(drop=True)

    _, _, mid = _vegalite_angles(df[value_col])

    sx_orig = np.sin(mid)
    sy_orig = -np.cos(mid)

    text_mid = mid.copy().tolist()
    min_gap = np.deg2rad(13.0)
    max_drift = np.deg2rad(40.0)

    for _ in range(300):
        moved = False

        for i in range(len(text_mid)):
            j = (i + 1) % len(text_mid)

            diff = (text_mid[j] - text_mid[i]) % (2 * np.pi)

            if 0 < diff < min_gap:
                push = (min_gap - diff) / 2

                new_i = text_mid[i] - push
                new_j = text_mid[j] + push

                drift_i = abs((new_i - mid[i] + np.pi) % (2 * np.pi) - np.pi)
                drift_j = abs((new_j - mid[j] + np.pi) % (2 * np.pi) - np.pi)

                if drift_i <= max_drift:
                    text_mid[i] = new_i

                if drift_j <= max_drift:
                    text_mid[j] = new_j

                moved = True

        if not moved:
            break

    sx_text = np.sin(text_mid)
    sy_text = -np.cos(text_mid)

    elbow_r = outer_radius + radial_line_len
    text_r = outer_radius + radial_line_len + horizontal_line_len

    df["_x0"] = cx + outer_radius * sx_orig
    df["_y0"] = cy + outer_radius * sy_orig

    df["_x1"] = cx + elbow_r * sx_orig
    df["_y1"] = cy + elbow_r * sy_orig

    df["_x2"] = cx + text_r * sx_text
    df["_y2"] = cy + text_r * sy_text

    df["_side"] = np.where(sx_text >= 0, "right", "left")

    df["_label_x"] = np.where(
        sx_text >= 0,
        df["_x2"] + label_pad,
        df["_x2"] - label_pad,
    )
    df["_label_y"] = df["_y2"]

    return df


def build_altair_label_layer(
    label_df,
    text_col,
    colors_col,
    position,
    chart_width=400,
    chart_height=400,
    font_size=10,
    use_leaders=False,
):
    """
    Draw label text, and optionally leader lines.

    This layer uses x/y coordinates because we want custom placement,
    but the coordinates now match the arc order exactly.
    """
    label_df = label_df.copy()
    mask = label_df[text_col] != ""

    if use_leaders:
        lc = "#666666"

        radial = alt.Chart(label_df[mask]).mark_rule(color=lc).encode(
            x=alt.X("_x0:Q", scale=alt.Scale(domain=[0, chart_width]), axis=None),
            y=alt.Y("_y0:Q", scale=alt.Scale(domain=[chart_height, 0]), axis=None),
            x2=alt.X2("_x1:Q"),
            y2=alt.Y2("_y1:Q"),
        )

        horizontal = alt.Chart(label_df[mask]).mark_rule(color=lc).encode(
            x=alt.X("_x1:Q", scale=alt.Scale(domain=[0, chart_width]), axis=None),
            y=alt.Y("_y1:Q", scale=alt.Scale(domain=[chart_height, 0]), axis=None),
            x2=alt.X2("_x2:Q"),
            y2=alt.Y2("_y2:Q"),
        )

        right = alt.Chart(
            label_df[mask & (label_df["_side"] == "right")]
        ).mark_text(
            align="left",
            baseline="middle",
            size=font_size,
        ).encode(
            x=alt.X("_label_x:Q", scale=alt.Scale(domain=[0, chart_width]), axis=None),
            y=alt.Y("_label_y:Q", scale=alt.Scale(domain=[chart_height, 0]), axis=None),
            text=alt.Text(f"{text_col}:N"),
            color=alt.Color(f"{colors_col}:N", scale=None, legend=None),
        )

        left = alt.Chart(
            label_df[mask & (label_df["_side"] == "left")]
        ).mark_text(
            align="right",
            baseline="middle",
            size=font_size,
        ).encode(
            x=alt.X("_label_x:Q", scale=alt.Scale(domain=[0, chart_width]), axis=None),
            y=alt.Y("_label_y:Q", scale=alt.Scale(domain=[chart_height, 0]), axis=None),
            text=alt.Text(f"{text_col}:N"),
            color=alt.Color(f"{colors_col}:N", scale=None, legend=None),
        )

        return radial + horizontal + right + left

    base = alt.Chart(label_df[mask]).encode(
        x=alt.X("_label_x:Q", scale=alt.Scale(domain=[0, chart_width]), axis=None),
        y=alt.Y("_label_y:Q", scale=alt.Scale(domain=[chart_height, 0]), axis=None),
        text=alt.Text(f"{text_col}:N"),
        color=alt.Color(f"{colors_col}:N", scale=None, legend=None),
    )

    if position == "inside":
        return base.mark_text(
            size=font_size,
            baseline="middle",
            align="center",
        )

    right = base.transform_filter(
        alt.datum._cos >= 0
    ).mark_text(
        size=font_size,
        baseline="middle",
        align="left",
    )

    left = base.transform_filter(
        alt.datum._cos < 0
    ).mark_text(
        size=font_size,
        baseline="middle",
        align="right",
    )

    return right + left


def render_pie_altair(plot_df, category_col, value_col, title, subtitle, style, rng=None):
    plot_df = apply_sort_order(plot_df, value_col, style, rng=rng).copy()
    plot_df = plot_df.reset_index(drop=True)

    # This is the important ordering column.
    # Every arc and every label uses this same order.
    plot_df["_pie_order"] = np.arange(len(plot_df))

    style = sanitize_style_for_altair(plot_df, style)
    colors = build_palette(len(plot_df), style, rng=rng)

    plot_df["_color"] = colors

    label_df = build_label_plan(plot_df, category_col, value_col, style).copy()
    label_df = label_df.reset_index(drop=True)

    # Carry the exact same pie order into the label dataframe.
    label_df["_pie_order"] = plot_df["_pie_order"].values

    label_df["_inside_color"] = [
        resolve_text_color(style["label_text_color"], "inside", fill_color, style)
        for fill_color in colors
    ]

    label_df["_outside_color"] = [
        resolve_text_color(style["label_text_color"], "outside", fill_color, style)
        for fill_color in colors
    ]

    total = float(plot_df[value_col].sum())

    if total <= 0:
        shares = np.ones(len(plot_df)) / len(plot_df)
    else:
        shares = (plot_df[value_col] / total).values

    # Suppress tiny labels to avoid unreadable overlaps.
    for col in ["_inside_text", "_outside_text"]:
        label_df[col] = [
            text if share >= 0.04 else ""
            for text, share in zip(label_df[col], shares)
        ]

    title_fontsize = get_title_fontsize(style)
    legend_orient = style.get("legend_orientation", "none")
    has_title = bool(style.get("title_present"))
    has_subtitle = bool(style.get("subtitle_present")) and subtitle

    top_legend_orientations = {"top", "top_left", "top_right"}
    extra_top_padding = 0

    if has_title and legend_orient in top_legend_orientations:
        n_items = len(plot_df)
        row_h = 18
        legend_h = n_items * row_h
        extra_top_padding = max(0, legend_h - 20)

    title_kwargs = {}

    if has_title:
        title_kwargs["title"] = alt.TitleParams(
            text=title,
            fontSize=title_fontsize,
            color=style.get("title_color", "black"),
            anchor=get_altair_title_anchor(style),
            subtitle=[subtitle] if has_subtitle else alt.Undefined,
            offset=extra_top_padding + 6,
        )

    bg = get_background_color(style)
    stroke_color, stroke_width = resolve_border_settings(style)

    base = alt.Chart(
        plot_df,
        **title_kwargs,
    ).properties(
        width=400,
        height=400,
    )

    arc = base.mark_arc(
        outerRadius=160,
        innerRadius=0,
        stroke=stroke_color,
        strokeWidth=stroke_width,
    ).encode(
        theta=alt.Theta(
            field=value_col,
            type="quantitative",
        ),

        # Critical fix:
        # force Vega-Lite's arc stack order to match our Python label geometry.
        order=alt.Order(
            field="_pie_order",
            type="quantitative",
            sort="ascending",
        ),

        color=alt.Color(
            field=category_col,
            type="nominal",
            scale=alt.Scale(
                domain=plot_df[category_col].tolist(),
                range=colors,
            ),
            legend=alt.Legend(
                orient=legend_orient.replace("_", "-"),
                title=get_legend_title_text(category_col, style) or alt.Undefined,
                labelColor=get_legend_text_color(style) or "black",
                titleColor=style.get("legend_title_color") or "black",
                titleFontSize=max(10, title_fontsize - 2),
                strokeColor="black" if style.get("legend_outline") else alt.Undefined,
                padding=6 if style.get("legend_outline") or style.get("legend_fill") == "gray_block" else 0,
                fillColor=style["legend_fill"] if style.get("legend_fill") not in {"none", None} else alt.Undefined,
            ) if style.get("legend_present") else None,
        ),
        tooltip=[category_col, value_col],
    )

    layers = [arc]

    if style.get("label_placement") != "none" and style.get("slice_text_content") != "none":
        if (label_df["_inside_text"] != "").any():
            inside_geom = compute_altair_text_geometry(
                label_df,
                value_col,
                position="inside",
            )

            layers.append(
                build_altair_label_layer(
                    inside_geom,
                    "_inside_text",
                    "_inside_color",
                    "inside",
                    font_size=10,
                )
            )

        if (label_df["_outside_text"] != "").any():
            use_leaders = bool(label_df["_outside_with_leader"].any())

            if use_leaders:
                outside_geom = compute_altair_leaderline_geometry(
                    label_df,
                    value_col,
                )
            else:
                outside_geom = compute_altair_text_geometry(
                    label_df,
                    value_col,
                    position="outside",
                )

            layers.append(
                build_altair_label_layer(
                    outside_geom,
                    "_outside_text",
                    "_outside_color",
                    "outside",
                    font_size=10,
                    use_leaders=use_leaders,
                )
            )

    padding = (
        {
            "top": extra_top_padding + title_fontsize + 20,
            "left": 10,
            "right": 10,
            "bottom": 10,
        }
        if extra_top_padding > 0
        else 10
    )

    return alt.layer(*layers).configure_view(
        stroke="black" if style.get("image_outline") else None,
        fill=None if bg == "none" else bg,
    ).configure(
        padding=padding,
    )

In [13]:
def get_matplotlib_text_rotation(mid_angle_deg, orientation):
    if orientation == "horizontal":
        return 0.0
    if orientation in {"radial", "mixed"}:
        r = mid_angle_deg
        if 90 < r < 270:
            r += 180
        while r > 180:
            r -= 360
        while r <= -180:
            r += 360
        return r
    return 0.0


def add_outside_labels_with_leader_lines_matplotlib(
    ax,
    wedges,
    text_values,
    text_colors,
    font_size,
    orientation="horizontal",
    radius=1.0,
    text_radius=1.14,
):
    """
    Straight leader lines with shorter distance from pie to label.

    Important:
    - text_radius is a fixed default here
    - dynamic values based on n are passed from render_pie_matplotlib()
    """
    def estimate_y_for_side(side_items, y_nat, min_gap, y_limit):
        y = min(y_nat, y_limit)

        for prev in side_items:
            if y > prev["y_text"] - min_gap:
                y = prev["y_text"] - min_gap

        y = max(y, -y_limit)
        return y

    def insert_sorted(side_items, item):
        side_items.append(item)
        side_items.sort(key=lambda d: d["y_text"], reverse=True)

    active = []

    for wedge, label, color in zip(wedges, text_values, text_colors):
        if not label:
            continue

        angle = 0.5 * (wedge.theta1 + wedge.theta2)
        angle_rad = np.deg2rad(angle)

        x_edge = np.cos(angle_rad) * radius
        y_edge = np.sin(angle_rad) * radius
        y_nat = np.sin(angle_rad) * text_radius

        preferred_side = "right" if x_edge >= 0 else "left"

        active.append({
            "label": label,
            "color": color,
            "angle": angle,
            "angle_rad": angle_rad,
            "x_edge": x_edge,
            "y_edge": y_edge,
            "y_nat": y_nat,
            "preferred_side": preferred_side,
        })

    if not active:
        return

    # Slices very close to the top/bottom seam can be assigned to either side.
    # This helps tiny clustered slices avoid fighting for the same side.
    seam_band = radius * 0.22

    min_gap = max(0.09, font_size * 0.010)
    y_limit = max(1.02, text_radius * 0.92)

    switch_penalty = 0.08

    left_items = []
    right_items = []

    # Place labels from top to bottom.
    active.sort(key=lambda d: d["y_nat"], reverse=True)

    for item in active:
        x_edge = item["x_edge"]
        preferred = item["preferred_side"]

        if abs(x_edge) <= seam_band:
            allowed_sides = ["left", "right"]
        else:
            allowed_sides = [preferred]

        best_choice = None

        for side in allowed_sides:
            side_items = right_items if side == "right" else left_items
            y_est = estimate_y_for_side(side_items, item["y_nat"], min_gap, y_limit)

            cost = abs(y_est - item["y_nat"])

            if side != preferred:
                cost += switch_penalty

            # Small crowding penalty.
            cost += 0.015 * len(side_items)

            if best_choice is None or cost < best_choice["cost"]:
                best_choice = {
                    "side": side,
                    "y_text": y_est,
                    "cost": cost,
                }

        item["side"] = best_choice["side"]
        item["y_text"] = best_choice["y_text"]

        if item["side"] == "right":
            insert_sorted(right_items, item)
        else:
            insert_sorted(left_items, item)

    placed_items = left_items + right_items

    text_pad = 0.015

    for item in placed_items:
        sign = 1 if item["side"] == "right" else -1

        x0 = item["x_edge"]
        y0 = item["y_edge"]

        x_text = sign * text_radius
        y_text = item["y_text"]

        # Straight, short leader line.
        ax.plot([x0, x_text], [y0, y_text], color="gray", lw=0.75)

        ha = "left" if sign > 0 else "right"
        rotation = 0 if orientation == "horizontal" else get_matplotlib_text_rotation(item["angle"], "radial")

        label_color = item["color"]
        try:
            if relative_luminance(label_color) > 0.85:
                label_color = "black"
        except Exception:
            label_color = "black"

        ax.text(
            x_text + sign * text_pad,
            y_text,
            item["label"],
            ha=ha,
            va="center",
            rotation=rotation,
            rotation_mode="anchor",
            fontsize=font_size,
            color=label_color,
            clip_on=False,
        )

    # Shorter padding because text_radius is now closer to the pie.
    pad = text_radius + 0.04
    ax.set_xlim(-pad, pad)
    ax.set_ylim(-pad, pad)
    
def get_matplotlib_layout_for_legend(style, n, has_outside_labels=False):
    """
    Reserve figure space for legend so it does not overlap the pie or labels.
    Returns:
    - figsize
    - subplot_adjust kwargs
    - legend loc
    - legend bbox_anchor
    - legend ncol
    """
    orient = style.get("legend_orientation", "none")
    legend_present = bool(style.get("legend_present")) and orient != "none"

    figsize = (8, 6) if n <= 5 else (10, 7) if n <= 8 else (12, 8)

    if not legend_present:
        return figsize, dict(left=0.08, right=0.95, top=0.88, bottom=0.10), None, None, 1

    # Outside labels need extra room around the pie.
    label_extra = 0.05 if has_outside_labels else 0.00

    if orient == "right":
        return (
            figsize,
            dict(left=0.06, right=0.62 - label_extra, top=0.88, bottom=0.10),
            "center left",
            (1.02, 0.5),
            1,
        )

    if orient == "left":
        return (
            figsize,
            dict(left=0.38 + label_extra, right=0.96, top=0.88, bottom=0.10),
            "center right",
            (-0.02, 0.5),
            1,
        )

    if orient == "top":
        return (
            figsize,
            dict(left=0.08, right=0.95, top=0.66 - label_extra, bottom=0.10),
            "lower center",
            (0.5, 1.03),
            min(n, 5),
        )

    if orient == "bottom":
        return (
            figsize,
            dict(left=0.08, right=0.95, top=0.88, bottom=0.26 + label_extra),
            "upper center",
            (0.5, -0.08),
            min(n, 5),
        )

    if orient == "top_left":
        return (
            figsize,
            dict(left=0.34 + label_extra, right=0.96, top=0.76, bottom=0.10),
            "upper right",
            (-0.02, 1.02),
            1,
        )

    if orient == "top_right":
        return (
            figsize,
            dict(left=0.06, right=0.66 - label_extra, top=0.76, bottom=0.10),
            "upper left",
            (1.02, 1.02),
            1,
        )

    return figsize, dict(left=0.08, right=0.95, top=0.88, bottom=0.10), None, None, 1


def render_pie_matplotlib(plot_df, category_col, value_col, title, subtitle, style, rng=None):
    plot_df = apply_sort_order(plot_df, value_col, style, rng=rng).copy()
    style = sanitize_style_for_matplotlib(plot_df, style)
    colors = build_palette(len(plot_df), style, rng=rng)
    label_df = build_label_plan(plot_df, category_col, value_col, style)

    n = len(plot_df)

    has_outside_labels = (
        (label_df["_outside_text"] != "").any()
        and style.get("label_placement") != "none"
        and style.get("slice_text_content") != "none"
    )

    figsize, subplot_adjust, legend_loc, legend_bbox, legend_ncol = get_matplotlib_layout_for_legend(
        style,
        n,
        has_outside_labels=has_outside_labels,
    )

    bg = get_background_color(style)
    facecolor = "none" if bg == "none" else bg

    fig, ax = plt.subplots(figsize=figsize, facecolor=facecolor)
    ax.set_facecolor(facecolor)

    plt.subplots_adjust(**subplot_adjust)

    border_color, border_width = resolve_border_settings(style)

    wedgeprops = (
        {"edgecolor": border_color, "linewidth": border_width}
        if border_width > 0 and border_color
        else None
    )

    radius = 1.0 if n <= 8 else 0.92

    wedges, _ = ax.pie(
        plot_df[value_col],
        labels=None,
        colors=colors,
        wedgeprops=wedgeprops,
        startangle=90,
        radius=radius,
    )

    inside_colors = [
        resolve_text_color(style["label_text_color"], "inside", fill_color, style)
        for fill_color in colors
    ]

    outside_colors = [
        resolve_text_color(style["label_text_color"], "outside", fill_color, style)
        for fill_color in colors
    ]

    total = float(plot_df[value_col].sum())

    if total <= 0:
        shares = np.ones(len(plot_df)) / len(plot_df)
    else:
        shares = (plot_df[value_col] / total).values

    min_inside_share = 0.05 if n <= 5 else 0.08

    if (label_df["_inside_text"] != "").any():
        for wedge, text_val, text_color_base, boxed, share in zip(
            wedges,
            label_df["_inside_text"],
            inside_colors,
            label_df["_boxed_text"],
            shares,
        ):
            if not text_val or share < min_inside_share:
                continue

            angle = 0.5 * (wedge.theta1 + wedge.theta2)
            angle_rad = np.deg2rad(angle)

            x = np.cos(angle_rad) * radius * 0.58
            y = np.sin(angle_rad) * radius * 0.58

            if boxed or style.get("label_text_color") == "white_on_black_box":
                box_fill = (
                    "black"
                    if style.get("label_text_color") == "white_on_black_box"
                    else colors[list(wedges).index(wedge)]
                )

                text_color = resolve_text_color(
                    style["label_text_color"],
                    "inside_boxed",
                    box_fill,
                    style,
                )

                bbox = {
                    "boxstyle": "round,pad=0.25",
                    "facecolor": box_fill,
                    "edgecolor": "none",
                }
            else:
                text_color = text_color_base
                bbox = None

            ax.text(
                x,
                y,
                text_val,
                ha="center",
                va="center",
                color=text_color,
                fontsize=10 if n <= 6 else 8,
                rotation=get_matplotlib_text_rotation(
                    angle,
                    style.get("slice_text_orientation", "horizontal"),
                ),
                rotation_mode="anchor",
                bbox=bbox,
            )

    if (label_df["_outside_text"] != "").any():
        outside_texts = label_df["_outside_text"].tolist()

        # Optional but recommended:
        # hide extremely tiny direct labels so leader lines stay short and readable.
        # These categories still appear in the legend.
        outside_texts = [
            text if share >= 0.01 else ""
            for text, share in zip(outside_texts, shares)
        ]

        if any(outside_texts):
            add_outside_labels_with_leader_lines_matplotlib(
                ax,
                wedges,
                outside_texts,
                outside_colors,
                font_size=9 if n <= 6 else 8,
                orientation="horizontal",
                radius=radius,
                text_radius=1.05 if n <= 6 else 1.08,
            )

    orient = style.get("legend_orientation", "none")
    legend = None

    if (
        style.get("legend_present")
        and orient != "none"
        and legend_loc is not None
        and legend_bbox is not None
    ):
        legend_kwargs = {
            "handles": wedges,
            "labels": plot_df[category_col].tolist(),
            "title": get_legend_title_text(category_col, style),
            "frameon": style.get("legend_outline", False),
        }

        legend = ax.legend(
            loc=legend_loc,
            bbox_to_anchor=legend_bbox,
            ncol=legend_ncol,
            borderaxespad=0.0,
            **legend_kwargs,
        )

        txt_color = get_legend_text_color(style)

        if txt_color:
            for txt in legend.get_texts():
                txt.set_color(txt_color)

        if legend.get_title():
            legend.get_title().set_color(style.get("legend_title_color") or "black")
            legend.get_title().set_fontsize(max(10, get_title_fontsize(style) - 2))

        fill = style.get("legend_fill", "none")

        if fill == "gray_block":
            fill = "#eeeeee"

        if fill not in {"none", None}:
            frame = legend.get_frame()
            frame.set_facecolor(fill)
            frame.set_edgecolor("black" if style.get("legend_outline") else fill)

    title_x, title_ha = get_title_alignment(style)
    title_obj = None
    subtitle_obj = None

    if style.get("title_present"):
        title_obj = fig.suptitle(
            title,
            x=title_x,
            ha=title_ha,
            y=0.97,
            color=style.get("title_color", "black"),
            fontsize=get_title_fontsize(style),
        )

    if style.get("subtitle_present") and subtitle:
        subtitle_obj = fig.text(
            title_x,
            0.93,
            subtitle,
            ha=title_ha,
            va="center",
            fontsize=max(9, get_title_fontsize(style) - 4),
            color="gray",
        )

    # Push title up if any outside labels overlap it.
    if title_obj is not None:
        fig.canvas.draw()

        renderer = fig.canvas.get_renderer()
        inv = fig.transFigure.inverted()

        title_bb = title_obj.get_window_extent(renderer=renderer)
        title_y0 = inv.transform((title_bb.x0, title_bb.y0))[1]

        highest_label_top = None

        for artist in ax.get_children():
            if not isinstance(artist, plt.Text):
                continue
            if not artist.get_text():
                continue

            try:
                bb = artist.get_window_extent(renderer=renderer)
                label_top = inv.transform((bb.x0, bb.y1))[1]

                if highest_label_top is None or label_top > highest_label_top:
                    highest_label_top = label_top

            except Exception:
                continue

        if highest_label_top is not None and highest_label_top > title_y0 - 0.01:
            gap = highest_label_top + 0.02 - title_y0
            new_y = title_obj.get_position()[1] + gap

            title_obj.set_position((title_x, new_y))

            if subtitle_obj is not None:
                old_sub_y = subtitle_obj.get_position()[1]
                subtitle_obj.set_position((title_x, old_sub_y + gap))

    outline_width = style.get("outline_chart", 0)

    if outline_width:
        ax.add_patch(
            Circle(
                (0, 0),
                radius=radius,
                fill=False,
                linewidth=float(outline_width),
                edgecolor="black",
            )
        )

    if style.get("image_outline", False):
        fig.patch.set_edgecolor("black")
        fig.patch.set_linewidth(2.0)

    return fig


def render_pie_seaborn(plot_df, category_col, value_col, title, subtitle, style, rng=None):
    if sns is None:
        raise ImportError("seaborn is not installed")
    original_build_palette = globals()["build_palette"]

    def seaborn_palette(n, style, rng=None):
        palette_name = {
            "bright_multicolor": "bright", "soft_pastel": "pastel",
            "single_color": "Blues", "grayscale": "gray",
        }.get(style.get("palette_type"), "deep")
        return sns.color_palette(palette_name, n_colors=n).as_hex()

    globals()["build_palette"] = seaborn_palette
    try:
        with sns.axes_style("white"):
            return render_pie_matplotlib(plot_df, category_col, value_col, title, subtitle, style, rng=rng)
    finally:
        globals()["build_palette"] = original_build_palette

def render_pie_plotly(plot_df, category_col, value_col, title, subtitle, style, rng=None):
    plot_df = apply_sort_order(plot_df, value_col, style, rng=rng).copy()
    style   = sanitize_style_for_plotly(plot_df, style)
    colors  = build_palette(len(plot_df), style, rng=rng)
    label_df = build_label_plan(plot_df, category_col, value_col, style)

    placement = style.get("label_placement", "none")
    plot_df["_display_text"] = ""

    if placement in {"inside", "mixed", "fit_inside_else_outside",
                     "split_label_outside_percent_inside", "split_label_outside_with_leader_lines_value_inside"}:
        plot_df["_display_text"] = np.where(label_df["_inside_text"] != "", label_df["_inside_text"], label_df["_outside_text"])
        text_position = "auto" if placement in {"mixed", "fit_inside_else_outside"} and (label_df["_outside_text"] != "").any() else "inside"
    elif placement == "outside_with_leader_lines":
        plot_df["_display_text"] = label_df["_outside_text"]
        text_position = "outside"
        #fig_kwargs["pull"] = [0.05] * len(plot_df)  # slight pull forces Plotly to draw leader lines
    elif placement == "outside_without_leader_lines":
        plot_df["_display_text"] = label_df["_outside_text"]
        text_position = "outside"
    else:
        text_position = "none"

    def plotly_text_mode(content):
        if content in {"percent", "percent_box"}: return "percent"
        if content == "label": return "label"
        if content in {"label_percent_inline", "label_percent_stacked", "percent_above_label"}: return "label+percent"
        if content == "value": return "value"
        return "text"

    fig_kwargs = {"data_frame": plot_df, "names": category_col, "values": value_col, "color_discrete_sequence": colors}
    if style.get("title_present"): fig_kwargs["title"] = title
    fig = px.pie(**fig_kwargs)

    border_color, border_width = resolve_border_settings(style)
    fig.update_traces(marker=dict(line=dict(color=border_color, width=border_width)))

    if placement == "none" or style.get("slice_text_content") == "none":
        fig.update_traces(textinfo="none")
    else:
        textinfo = plotly_text_mode(style.get("slice_text_content", "none"))
        if textinfo == "text":
            fig.update_traces(textinfo="text", text=plot_df["_display_text"], textposition=text_position)
        else:
            fig.update_traces(textinfo=textinfo, textposition=text_position)

    color_mode = style.get("label_text_color", "black")
    if color_mode != "none":
        per_slice_colors = [
            resolve_text_color(color_mode, "inside", fill_color, style)
            for fill_color in colors
        ]
        # Plotly accepts a list of colors mapped per slice
        fig.update_traces(textfont=dict(color=per_slice_colors))

    title_x, title_anchor = get_plotly_title_anchor(style)
    bg = get_background_color(style)
    has_title    = bool(style.get("title_present"))
    has_subtitle = bool(style.get("subtitle_present")) and bool(subtitle)
    orient       = style.get("legend_orientation", "none")
    title_fs     = get_title_fontsize(style)

    # --- Overlap-safe margin ---
    top_legend_orientations = {"top", "top_left", "top_right"}

    if orient in top_legend_orientations and style.get("legend_present"):
        n_items     = len(plot_df)
        legend_px   = n_items * 19 + 10
        title_px    = title_fs + 6
        subtitle_px = (max(9, title_fs - 4) + 4) if has_subtitle else 0
        margin_top  = max(100, legend_px + title_px + subtitle_px + 24)
    else:
        title_px    = title_fs + 6
        subtitle_px = (max(9, title_fs - 4) + 4) if has_subtitle else 0
        margin_top  = 90 if has_title else 40

    # --- Build title text with subtitle embedded as HTML second line ---
    if has_title and has_subtitle:
        subtitle_fs = max(9, title_fs - 4)
        title_text = (
            f'{title}<br>'
            f'<span style="font-size:{subtitle_fs}px; color:gray;">{subtitle}</span>'
        )
    elif has_title:
        title_text = title
    else:
        title_text = ""

    def get_plotly_legend_layout(style):
        orient = style.get("legend_orientation", "right")
        layout = {
            "font":        {"color": get_legend_text_color(style) or "black"},
            "title":       {"text": "", "font": {"color": style.get("legend_title_color") or "black",
                                                  "size": max(10, get_title_fontsize(style) - 2)}},
            "bordercolor": "black" if style.get("legend_outline") else "rgba(0,0,0,0)",
            "borderwidth": 1 if style.get("legend_outline") else 0,
            "bgcolor": style.get("legend_fill") if style.get("legend_fill") not in {"none", None} else "rgba(0,0,0,0)",
        }
        pos_map = {
            "right":     {"orientation": "v", "x": 1.02,  "y": 0.5,  "xanchor": "left",   "yanchor": "middle"},
            "left":      {"orientation": "v", "x": -0.02, "y": 0.5,  "xanchor": "right",  "yanchor": "middle"},
            "top":       {"orientation": "h", "x": 0.5,   "y": 1.0,  "xanchor": "center", "yanchor": "bottom"},
            "bottom":    {"orientation": "h", "x": 0.5,   "y": -0.08,"xanchor": "center", "yanchor": "top"},
            "top_left":  {"orientation": "v", "x": 0.00,  "y": 1.0,  "xanchor": "left",   "yanchor": "bottom"},
            "top_right": {"orientation": "v", "x": 1.00,  "y": 1.0,  "xanchor": "right",  "yanchor": "bottom"},
        }
        if orient in pos_map: layout.update(pos_map[orient])
        return layout

    fig.update_layout(
        showlegend=bool(style.get("legend_present")),
        title=dict(
            text=title_text,
            x=title_x, xanchor=title_anchor,
            y=0.99, yanchor="top",
            font=dict(size=title_fs, color=style.get("title_color", "black")),
            pad=dict(b=10),
        ),
        legend=get_plotly_legend_layout(style),
        paper_bgcolor="rgba(0,0,0,0)" if bg == "none" else bg,
        plot_bgcolor="rgba(0,0,0,0)"  if bg == "none" else bg,
        margin=dict(l=40, r=40, t=margin_top, b=40),
    )
    legend_title_text = get_legend_title_text(category_col, style) or ""
    fig.update_layout(legend_title_text=legend_title_text)

    if style.get("image_outline"):
        fig.update_layout(shapes=[dict(type="rect", xref="paper", yref="paper",
                                       x0=0, y0=0, x1=1, y1=1, fillcolor="rgba(0,0,0,0)",
                                       line=dict(color="black", width=2))])
    return fig


## 6. Saving and generation

In [14]:
def new_chart_id(prefix: str = "pie") -> str:
    return f"{prefix}_{datetime.utcnow().strftime('%Y%m%dT%H%M%S')}_{uuid4().hex[:8]}"


def save_metadata(meta: dict, path: Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2, default=str)


def ensure_output_dirs(out_root: Path) -> None:
    if CLEAR_OUTPUT and out_root.exists():
        shutil.rmtree(out_root)
    for sub in SUBDIRS:
        for lib in LIBRARIES:
            (out_root / sub / lib).mkdir(parents=True, exist_ok=True)


def save_altair_svg(chart, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    chart.save(str(out_path), format="svg")


def save_plotly_png(fig, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.write_image(str(out_path), format="png", scale=2)


def save_matplotlib_png(fig, out_path: Path) -> None:
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=200, bbox_inches="tight",
                transparent=(fig.get_facecolor()[-1] == 0
                             if hasattr(fig.get_facecolor(), "__len__") else False))
    plt.close(fig)


def generate_pie(
    out_root: Path,
    dataset_source: str,
    library: str,
    rng_seed: int,
    param_stats: dict,
    datasets: dict | None = None,
    max_tries: int = 50,
) -> dict:
    rng = np.random.default_rng(rng_seed)
    chart_id = new_chart_id("pie")

    for attempt in range(1, max_tries + 1):
        style = harmonize_style(sample_style(rng, param_stats))

        plot_df = None
        context = None

        if datasets:
            ds_names = list(datasets.keys())
            ds_name  = ds_names[int(rng.integers(0, len(ds_names)))]
            df       = datasets[ds_name]
            spec     = DATASET_REGISTRY[ds_name]
            result   = sample_pie_data_from_df(df, spec, rng, style)
            if result is not None:
                plot_df, context = result
                dataset_source = ds_name
            else:
                print(f"[seed {rng_seed}] Real data sampling failed on attempt {attempt}")

        if plot_df is None:
            continue

        category_col = context["category_col"]
        value_col    = context["value_col"]
        title, subtitle = make_title(context, style=style)

        table_path = out_root / "tables"   / library / f"{chart_id}.csv"
        meta_path  = out_root / "metadata" / library / f"{chart_id}.json"
        plot_df.to_csv(table_path, index=False)

        if library == "altair":
            image_path = out_root / "images" / library / f"{chart_id}.svg"
            chart = render_pie_altair(plot_df, category_col, value_col, title, subtitle, style, rng=rng)
            save_altair_svg(chart, image_path)
        elif library == "matplotlib":
            image_path = out_root / "images" / library / f"{chart_id}.png"
            fig = render_pie_matplotlib(plot_df, category_col, value_col, title, subtitle, style, rng=rng)
            save_matplotlib_png(fig, image_path)
        elif library == "seaborn":
            image_path = out_root / "images" / library / f"{chart_id}.png"
            fig = render_pie_seaborn(plot_df, category_col, value_col, title, subtitle, style, rng=rng)
            save_matplotlib_png(fig, image_path)
        elif library == "plotly":
            image_path = out_root / "images" / library / f"{chart_id}.png"
            fig = render_pie_plotly(plot_df, category_col, value_col, title, subtitle, style, rng=rng)
            save_plotly_png(fig, image_path)
        else:
            raise ValueError(f"Unsupported library: {library}")

        meta = {
            "chart_id":       chart_id,
            "chart_type":     "pie",
            "library":        library,
            "dataset_source": dataset_source,
            "image_path":     str(image_path),
            "table_path":     str(table_path),
            "data_context":   context,
            "style":          style,
            "created_utc":    datetime.utcnow().isoformat() + "Z",
            "rng_seed":       rng_seed,
        }
        save_metadata(meta, meta_path)
        return meta

    raise RuntimeError(f"Failed to generate pie after {max_tries} attempts.")


def generate_batch(
    out_root: Path,
    dataset_source: str,
    generation_plan: dict,
    param_stats: dict,
    datasets: dict | None = None,
    start_seed: int = 1000,
) -> list[dict]:
    ensure_output_dirs(out_root)
    metas = []
    seed = start_seed
    for library, n in generation_plan.items():
        for _ in range(n):
            metas.append(generate_pie(
                out_root=out_root,
                dataset_source=dataset_source,
                library=library,
                rng_seed=seed,
                param_stats=param_stats,
                datasets=datasets,
            ))
            seed += 1
    return metas

In [15]:
ensure_output_dirs(OUT_ROOT)

weights = normalize_weights(SAMPLING_WEIGHTS)

generation_plan = {
    "altair":     10,
    "matplotlib": 10,
    "seaborn":    10,
    "plotly":     10,
}

metas = generate_batch(
    out_root=OUT_ROOT,
    dataset_source="warehouse_retail",
    generation_plan=generation_plan,
    param_stats=weights,
    datasets=DATASETS,
    start_seed=666,
)

pd.DataFrame(metas)[["chart_id", "library", "dataset_source", "image_path"]].head()

Resorting to unclean kill browser.


,chart_id,library,dataset_source,image_path
0,pie_20260614T161833_b2d70625,altair,warehouse_retail,C:\Users\Michelle\I2R\notebooks\final_notebook...
1,pie_20260614T161834_6280aa8b,altair,warehouse_retail,C:\Users\Michelle\I2R\notebooks\final_notebook...
2,pie_20260614T161835_a2e77732,altair,warehouse_retail,C:\Users\Michelle\I2R\notebooks\final_notebook...
3,pie_20260614T161835_e9b06608,altair,warehouse_retail,C:\Users\Michelle\I2R\notebooks\final_notebook...
4,pie_20260614T161836_d604d30e,altair,warehouse_retail,C:\Users\Michelle\I2R\notebooks\final_notebook...


# TESTING

### Smoke test

In [16]:
import re
from pathlib import Path

TEST_LIBRARIES = ["matplotlib", "seaborn", "altair", "plotly"]

base = {param: max(codes, key=codes.get) for param, codes in SAMPLING_WEIGHTS.items()}
base.update({
    "title_present": True, "title_location": "center", "title_color": "black", "title_size": "medium",
    "legend_present": True, "legend_orientation": "right", "slice_count": 5,
    "slice_text_present": True, "slice_text_content": "percent",
    "label_placement": "inside", "background": "white",
})

errors = []
rng = np.random.default_rng(42)

for param_name, val_probs in SAMPLING_WEIGHTS.items():
    for val in val_probs.keys():
        style = harmonize_style({**base, param_name: val})
        # Pick a dataset
        ds_name = list(DATASETS.keys())[0]
        df = DATASETS[ds_name]
        spec = DATASET_REGISTRY[ds_name]
        result = sample_pie_data_from_df(df, spec, rng, style)
        if result is None:
            continue
        plot_df, context = result
        category_col = context["category_col"]
        value_col = context["value_col"]
        title, subtitle = "Test", None

        for library in TEST_LIBRARIES:
            try:
                if library == "matplotlib":
                    fig = render_pie_matplotlib(plot_df, category_col, value_col, title, subtitle, style)
                    plt.close(fig)
                elif library == "seaborn":
                    fig = render_pie_seaborn(plot_df, category_col, value_col, title, subtitle, style)
                    plt.close(fig)
                elif library == "altair":
                    render_pie_altair(plot_df, category_col, value_col, title, subtitle, style)
                elif library == "plotly":
                    render_pie_plotly(plot_df, category_col, value_col, title, subtitle, style)
            except Exception as e:
                errors.append({"param": param_name, "value": val, "library": library,
                               "error": f"{type(e).__name__}: {e}"})

if errors:
    print(f"Found {len(errors)} errors:")
    pd.DataFrame(errors)
else:
    print("All parameter values passed for all libraries.")

Found 1 errors:
